# GPT-2 Fine-tuning (Causal Language Modeling on News Descriptions)

**GPT-2** is a *decoder-only*, autoregressive transformer: it predicts the next token given all
previous tokens. That makes it a **causal language model (LM)** — unlike BERT (encoder-only,
non-generative) and T5 (encoder-decoder, conditional generation).

Here we fine-tune GPT-2 on the Reuters **Description** text so it learns to *write* news-style
prose, then sample from it. Each description is wrapped in special `<|startoftext|>` /
`<|endoftext|>` tokens so the model learns where sequences begin and end.

The pipeline uses several training refinements exposed by `GPT2Config`: mixed precision (AMP),
gradient accumulation, gradient clipping, a warmup LR schedule, and early stopping.

> **Before you run this:** switch on the GPU with **Runtime → Change runtime type → Hardware
> accelerator → GPU (T4)**.

In [1]:
# Clone the repo (skips if it already exists)
![ -d /content/Bert-T5-GPT2 ] || git clone https://github.com/nickkats1/Bert-T5-GPT2

Cloning into 'Bert-T5-GPT2'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 75 (delta 17), reused 74 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 3.70 MiB | 9.40 MiB/s, done.
Resolving deltas: 100% (17/17), done.


## Install the package and its dependencies
Editable install pulls in `transformers`/`torch` and makes `import headlines...` work directly.

In [2]:
%pip install -q -e /content/Bert-T5-GPT2

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Building editable for bert-t5-gpt2 (pyproject.toml) ... done


In [6]:
import sys
sys.path.append('/content/Bert-T5-GPT2')

## Confirm the GPU is visible
AMP (mixed precision) only activates on CUDA, so a GPU is what makes this run fast.

In [7]:
import math, copy
import torch
from headlines.common import (
    resolve_device, seed_everything, count_parameters, make_generator, EarlyStopping,
)
from headlines.gpt2.config import CONFIG

seed_everything(CONFIG.seed)
device = resolve_device(CONFIG.device)
amp_enabled = CONFIG.use_amp and device.type == 'cuda'
print('Device:', device, '| AMP:', amp_enabled)
assert device.type == 'cuda', 'No GPU! Enable it via Runtime > Change runtime type.'

Device: cuda | AMP: True


## Load the descriptions and split them
For causal LM we only need the `Description` column. `load_data` drops `Time`/`Headlines`,
removes empty rows, and de-duplicates; `split_data` returns train/val **lists of strings**.

In [8]:
from headlines.gpt2.utils import load_data, split_data

description_df = load_data('/content/Bert-T5-GPT2/data/reuters_headlines.csv')
train_description, val_description = split_data(
    description_df, test_size=CONFIG.test_size, random_state=CONFIG.seed,
)
print(f'train={len(train_description):,}  val={len(val_description):,}')
train_description[0][:200]

train=26,059  val=6,515


'U.S. President Donald Trump, faced with mounting anger in the farm belt over policies that allow oil refineries to use less corn-based ethanol, summoned Cabinet members on Thursday to discuss ways to '

## Tokenizer and model
GPT-2 ships without pad/BOS/EOS tokens, so we add them and **resize the model's embedding table**
to match the enlarged vocabulary. We keep `padding_side='right'`: GPT-2 derives position ids from a
plain range, so left padding would shift every real token's positional embedding during training.

In [9]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained(CONFIG.model_name)
tokenizer.add_special_tokens(
    {'pad_token': CONFIG.pad_token, 'bos_token': CONFIG.bos_token, 'eos_token': CONFIG.eos_token}
)
tokenizer.padding_side = 'right'

model = GPT2LMHeadModel.from_pretrained(CONFIG.model_name)
model.resize_token_embeddings(len(tokenizer))
model.to(device)
print(f'Trainable parameters: {count_parameters(model):,}')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Trainable parameters: 124,441,344


## DataLoaders
`build_dataloaders` wraps each string in BOS/EOS and dynamically pads each batch to its longest
sequence (`max_length=128` cap). The training loader is shuffled with a seeded generator.

In [10]:
from headlines.gpt2.utils import build_dataloaders

train_loader, val_loader = build_dataloaders(
    train_description, val_description, tokenizer,
    batch_size=CONFIG.batch_size,
    max_length=CONFIG.max_length,
    generator=make_generator(CONFIG.seed),
)
batch = next(iter(train_loader))
{k: tuple(v.shape) for k, v in batch.items()}

{'input_ids': (12, 50), 'attention_mask': (12, 50)}

## Optimizer, LR schedule, and gradient scaler
We use `AdamW` with a **linear warmup** schedule. The number of optimizer steps must account for
gradient accumulation (the optimizer steps every `accum_steps` batches), so we compute
`steps_per_epoch = ceil(batches / accum_steps)`. The `GradScaler` handles AMP loss scaling.

In [11]:
from transformers import get_linear_schedule_with_warmup

optimizer = torch.optim.AdamW(
    model.parameters(), lr=CONFIG.learning_rate, weight_decay=CONFIG.weight_decay,
)
steps_per_epoch = math.ceil(len(train_loader) / max(1, CONFIG.accum_steps))
total_steps = max(1, steps_per_epoch * CONFIG.epochs)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(CONFIG.warmup_ratio * total_steps),
    num_training_steps=total_steps,
)
scaler = torch.amp.GradScaler(device.type, enabled=amp_enabled)

## Fine-tune with early stopping
Each epoch we `train` (returns loss + perplexity) then `validate`. `EarlyStopping` watches the
validation loss: we checkpoint the weights whenever it improves, and stop once it fails to improve
for `patience` epochs — so we keep the best model, not the last (possibly overfit) one.

> Padding is masked to `-100` in the loss, and **perplexity = exp(loss)** (lower is better).

In [12]:
from headlines.gpt2.trainer import train, validate

early_stopping = EarlyStopping(patience=CONFIG.patience, mode='min')
best_state = None

for epoch in range(CONFIG.epochs):
    train_loss, train_ppl = train(
        model, train_loader, optimizer, device,
        epoch=epoch, total_epochs=CONFIG.epochs,
        scheduler=scheduler, scaler=scaler,
        accum_steps=CONFIG.accum_steps, use_amp=CONFIG.use_amp,
        max_grad_norm=CONFIG.max_grad_norm,
    )
    val_loss, val_ppl = validate(model, val_loader, device)
    print(f'Epoch {epoch + 1}/{CONFIG.epochs} | '
          f'train ppl {train_ppl:.2f} | val ppl {val_ppl:.2f}')

    if early_stopping.step(val_loss):
        best_state = copy.deepcopy(model.state_dict())
    if early_stopping.should_stop:
        print(f'Early stopping (best val loss {early_stopping.best:.4f})')
        break

if best_state is not None:
    model.load_state_dict(best_state)  # restore best weights

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
21:15:21 | headlines.gpt2.trainer | INFO | Epoch [1/3] | Step [0/2172] | Loss: 4.2829
/content/Bert-T5-GPT2/headlines/gpt2/trainer.py:106: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning ra

Epoch 1/3 | train ppl 22.49 | val ppl 16.48


21:16:12 | headlines.gpt2.trainer | INFO | Epoch [2/3] | Step [10/2172] | Loss: 2.8798
21:16:12 | headlines.gpt2.trainer | INFO | Epoch [2/3] | Step [20/2172] | Loss: 2.8450
21:16:13 | headlines.gpt2.trainer | INFO | Epoch [2/3] | Step [30/2172] | Loss: 2.8560
21:16:13 | headlines.gpt2.trainer | INFO | Epoch [2/3] | Step [40/2172] | Loss: 2.8546
21:16:13 | headlines.gpt2.trainer | INFO | Epoch [2/3] | Step [50/2172] | Loss: 2.8598
21:16:13 | headlines.gpt2.trainer | INFO | Epoch [2/3] | Step [60/2172] | Loss: 2.8633
21:16:13 | headlines.gpt2.trainer | INFO | Epoch [2/3] | Step [70/2172] | Loss: 2.8509
21:16:14 | headlines.gpt2.trainer | INFO | Epoch [2/3] | Step [80/2172] | Loss: 2.8508
21:16:14 | headlines.gpt2.trainer | INFO | Epoch [2/3] | Step [90/2172] | Loss: 2.8653
21:16:14 | headlines.gpt2.trainer | INFO | Epoch [2/3] | Step [100/2172] | Loss: 2.8653
21:16:14 | headlines.gpt2.trainer | INFO | Epoch [2/3] | Step [110/2172] | Loss: 2.8674
21:16:14 | headlines.gpt2.trainer | INFO 

Epoch 2/3 | train ppl 16.53 | val ppl 15.54


21:17:03 | headlines.gpt2.trainer | INFO | Epoch [3/3] | Step [10/2172] | Loss: 2.7749
21:17:03 | headlines.gpt2.trainer | INFO | Epoch [3/3] | Step [20/2172] | Loss: 2.7716
21:17:03 | headlines.gpt2.trainer | INFO | Epoch [3/3] | Step [30/2172] | Loss: 2.7342
21:17:03 | headlines.gpt2.trainer | INFO | Epoch [3/3] | Step [40/2172] | Loss: 2.7066
21:17:03 | headlines.gpt2.trainer | INFO | Epoch [3/3] | Step [50/2172] | Loss: 2.7064
21:17:04 | headlines.gpt2.trainer | INFO | Epoch [3/3] | Step [60/2172] | Loss: 2.6988
21:17:04 | headlines.gpt2.trainer | INFO | Epoch [3/3] | Step [70/2172] | Loss: 2.7026
21:17:04 | headlines.gpt2.trainer | INFO | Epoch [3/3] | Step [80/2172] | Loss: 2.7105
21:17:04 | headlines.gpt2.trainer | INFO | Epoch [3/3] | Step [90/2172] | Loss: 2.7170
21:17:05 | headlines.gpt2.trainer | INFO | Epoch [3/3] | Step [100/2172] | Loss: 2.7165
21:17:05 | headlines.gpt2.trainer | INFO | Epoch [3/3] | Step [110/2172] | Loss: 2.7154
21:17:05 | headlines.gpt2.trainer | INFO 

Epoch 3/3 | train ppl 14.95 | val ppl 15.31


## Generate samples
Perplexity can hide degenerate repetition, so we sample a few unconditioned sequences (seeded only
with the BOS token) to eyeball whether the fine-tuned model writes fluent, on-domain text.

In [13]:
from headlines.gpt2.metrics import generate_samples

for i, sample in enumerate(generate_samples(model, tokenizer, device, num_samples=5), 1):
    print(f'{i}. {sample}\n')

1. The world's second-largest economy is "hopeful" the United States will deliver on its promise to cut its COVID-19 pandemic aid in the coming months, the International Monetary Fund said

2. The S&P 500 dipped to record low on Monday as investors weighed whether Wall Street's growing panic over the coronavirus threat will be enough to ease worries about the economy and a second U.

3. U.S. consumer confidence has surged to its highest level in a quarter in nearly four years, easing concerns that the coronavirus pandemic will take longer to spread, according to industry consultants and

4. China's top trade negotiator with the United States warned the United States on Wednesday that Washington is ignoring its warning about what could happen when China's trade surplus with the United States is slashed.

5. Saudi Arabia's state-owned oil company Aramco has suspended production at its oil facilities in Kuwait and Iraq, which were once considered crucial to its supply chain and for export